In [1]:
import os
import pandas as pd
import ast
import warnings

warnings.filterwarnings('ignore')

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
import requests
from io import StringIO
from Bio import SeqIO

COG_data_dir = '/active-data/datasets/COG_data'
os.makedirs(COG_data_dir, exist_ok=True)

if os.path.isfile(f'{COG_data_dir}/cog-24.def.tsv') and os.path.isfile(f'{COG_data_dir}/cog-24.fun.tsv'):
    COG_table = pd.read_csv(f'{COG_data_dir}/cog-24.def.tsv', sep='\t')
    COG_info = pd.read_csv(f'{COG_data_dir}/cog-24.fun.tsv', sep='\t')
else:
    url = r'https://ftp.ncbi.nlm.nih.gov/pub/COG/COG2024/data/cog-24.def.tab'
    response = requests.get(url)
    COG_table = pd.read_table(StringIO(response.text), header = None, names = ['COG_ID', 'Functional_Category', 'Gene_Name', 'Product', 'Pathway', 'PMID', 'PDB'])
    
    url = r'https://ftp.ncbi.nlm.nih.gov/pub/COG/COG2024/data/cog-24.fun.tab'
    cog_response = requests.get(url)
    COG_class = {}
    COG_info = pd.DataFrame()
    for line in cog_response.text.split('\n'):
        info_l = line.split('\t')
        if len(info_l) == 2:
            COG_class[info_l[0]] = info_l[1]
        elif len(info_l) > 2:
            temp_dict = {'char':info_l[0], 'class': COG_class[info_l[1]], 'description': info_l[-1]}
            COG_info = pd.concat([COG_info, pd.DataFrame([temp_dict])], ignore_index = True)
    COG_table.to_csv(f'{COG_data_dir}/cog-24.def.tsv', index=False, sep='\t')
    COG_info.to_csv(f'{COG_data_dir}/cog-24.fun.tsv', index=False, sep='\t')

In [3]:
from tqdm import tqdm
import numpy as np
import random
seed = 42
random.seed(seed)

head = 'query	seed_ortholog	evalue	score	eggNOG_OGs	max_annot_lvl	COG_category	Description	Preferred_name	GOs	EC	KEGG_ko	KEGG_Pathway	KEGG_Module	KEGG_Reaction	KEGG_rclass	BRITE	KEGG_TC	CAZy	BiGG_Reaction	PFAMs'.split('\t')
cog2cat = dict(zip(COG_table["COG_ID"], COG_table["Functional_Category"]))

def process_ogs(ogs_str):
    if pd.isna(ogs_str) or 'COG' not in ogs_str:
        return ''
    
    og_list = ogs_str.split(',')
    cog_set = set()
    for og in og_list:
        if 'COG' in og:
            cog = og.split('@')[0]
            cog_set.add(cog)
    
    cat_chars = set()
    for cog in cog_set:
        if cog in cog2cat:
            cat_chars.update(cog2cat[cog])
    
    return ''.join(sorted(cat_chars))

label = "pident_90"
size_thr = 10000
region_thr = 2500
base_folder = f'/active-data/analysis_results/chr_pla/genus'
for genus_name in keep_genus:
    replicon_data = pd.read_csv(f'{base_folder}/statistics_records/{genus_name}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    chromosomes = replicon_data[replicon_data[f'category-{label}']=='typical chromosome'].copy().reset_index(drop=True)
    IRs = replicon_data[replicon_data[f'category-{label}']=='intermediate replicon'].copy().reset_index(drop=True)
    
    all_cog_data = []
    random_cog_data = []
    with tqdm(total = len(IRs), desc=f'{genus_name}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for idx, row in IRs.iterrows():
            acc_n, contig = row['accession'].split('-')
            size = row['size']
            result_folder = f'/active-data/analysis_results/chr_pla/genus/cor-pla_fraction_records/{genus_name}/{acc_n}'
            blast_result = pd.read_csv(f'{result_folder}/{contig}_blast_result.csv')
            blast_result = blast_result[blast_result['pident'] >= 90].reset_index(drop=True)
            blast_result = blast_result[blast_result['sseqid'].isin(chromosomes['accession'])]
            blast_result = blast_result.sort_values('bitscore', ascending=False).reset_index(drop=True)
            if row['size'] > size_thr:
                selected_blast = blast_result.drop_duplicates(subset=['sseqid'], keep='first').head(5)
                for idx_s, row_s in selected_blast.iterrows():
                    acc_s, contig_s = row_s['sseqid'].split('-')
                    handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_s}/genomic.gbff')
                    acc_record = SeqIO.parse(handle, 'genbank')
                    CDSs = []
                    CDSs_ps = []
                    random_CDSs = []
                    random_CDSs_ps = []
                    for seq_record in acc_record:
                        if seq_record.id == contig_s:
                            start, end = min(row_s['sstart'], row_s['send']), max(row_s['sstart'], row_s['send'])
                            up_seq = seq_record[start-region_thr:start+region_thr]
                            down_seq = seq_record[end-region_thr:end+region_thr]
                            for dna_sequence in [up_seq, down_seq]:
                                for feature in dna_sequence.features:
                                    if feature.type == 'CDS':
                                        CDSs.append(feature.qualifiers['locus_tag'][0])
                                        if 'translation' in feature.qualifiers:
                                            CDSs_ps.append('False')
                                        else:
                                            CDSs_ps.append('True')
                                        
                            n = len(seq_record)
                            random_start = random.randint(region_thr, n-region_thr-abs(row_s['send']-row_s['sstart']))
                            random_end = random_start+abs(row_s['send']-row_s['sstart'])
                            random_up = seq_record[random_start-region_thr:random_start+region_thr]
                            random_down = seq_record[random_end-region_thr:random_end+region_thr]
                            for dna_sequence in [random_up, random_down]:
                                for feature in dna_sequence.features:
                                    if feature.type == 'CDS':
                                        random_CDSs.append(feature.qualifiers['locus_tag'][0])
                                        if 'translation' in feature.qualifiers:
                                            random_CDSs_ps.append('False')
                                        else:
                                            random_CDSs_ps.append('True')
                                        
                    eggnog_dir = f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/eggnog_results/{acc_s}'
                    eggnog_re = pd.read_csv(f'{eggnog_dir}/{acc_s}.emapper.annotations', comment = '#', sep = '\t', header = None, names = head)
                    eggnog_re['CDS'] = eggnog_re['query'].str.split('-').str[-1]
                    eggnog_re['COG_category_new'] = eggnog_re['eggNOG_OGs'].apply(process_ogs)
                    mask = (eggnog_re['COG_category_new'] != '') & (eggnog_re['COG_category_new'] != eggnog_re['COG_category'])
                    eggnog_re['COG_category_corrected'] = np.where(mask, eggnog_re['COG_category_new'], eggnog_re['COG_category'])

                    eggnog_selected = pd.DataFrame({"CDS": CDSs, "Psudogene": CDSs_ps})
                    eggnog_selected = pd.merge(eggnog_selected, eggnog_re, on='CDS', how='left')
                    eggnog_random = pd.DataFrame({"CDS": random_CDSs, "Psudogene": random_CDSs_ps})
                    eggnog_random = pd.merge(eggnog_random, eggnog_re, on='CDS', how='left')
                    all_cog_data.append(eggnog_selected)
                    random_cog_data.append(eggnog_random)
            pbar.update(1)
    if all_cog_data:
        all_cog_data = pd.concat(all_cog_data, ignore_index=True)
        #print(all_cog_data)
        target_folder = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
        all_cog_data.to_csv(f'{target_folder}/IRs_on_chromosome_gap_cog_result.tsv', sep='\t', index=False)
    else:
        print(f'No chromosomal data for {genus_name}!')
    
    if random_cog_data:
        random_cog_data = pd.concat(random_cog_data, ignore_index=True)
        #print(all_cog_data)
        target_folder = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
        random_cog_data.to_csv(f'{target_folder}/IRs_random_chromosome_gap_cog_result.tsv', sep='\t', index=False)
    else:
        print(f'No random data for {genus_name}!')

Klebsiella: 100%|███████████████████████████████████████████████████| 150/150 [05:34<00:00, 2.23s/B]
Staphylococcus: 100%|█████████████████████████████████████████████| 78.0/78.0 [01:10<00:00, 1.11B/s]
Pseudomonas: 100%|████████████████████████████████████████████████| 63.0/63.0 [02:02<00:00, 1.95s/B]
Salmonella: 100%|███████████████████████████████████████████████████| 109/109 [01:59<00:00, 1.09s/B]
Streptococcus: 100%|██████████████████████████████████████████████| 19.0/19.0 [00:15<00:00, 1.22B/s]
Streptomyces: 100%|███████████████████████████████████████████████| 64.0/64.0 [03:04<00:00, 2.88s/B]
Acinetobacter: 100%|██████████████████████████████████████████████| 69.0/69.0 [01:27<00:00, 1.27s/B]
Enterococcus: 100%|███████████████████████████████████████████████| 47.0/47.0 [00:42<00:00, 1.10B/s]
Bordetella: 100%|█████████████████████████████████████████████████| 2.00/2.00 [00:02<00:00, 1.09s/B]
Enterobacter: 100%|███████████████████████████████████████████████| 30.0/30.0 [01:05<00:00,

No chromosomal data for Corynebacterium!
No random data for Corynebacterium!


Burkholderia: 100%|███████████████████████████████████████████████| 8.00/8.00 [00:22<00:00, 2.77s/B]
Listeria: 100%|███████████████████████████████████████████████████| 2.00/2.00 [00:01<00:00, 1.25B/s]
Citrobacter: 100%|████████████████████████████████████████████████| 10.0/10.0 [00:18<00:00, 1.85s/B]
Helicobacter: 0.00B [00:00, ?B/s]

No chromosomal data for Helicobacter!
No random data for Helicobacter!
